<a href="https://colab.research.google.com/github/tomsBifx25/Mus-Glioblastoma-snRNAseq/blob/main/notebooks/Test_pipeV01_runs.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

New notebook with different approach

# **Predicting Malignant vs. Microenvironment Identity in Glioblastoma via Single-Nucleus Transcriptomics.**
****
*Thomas Walsh | March 18, 2026*

**Hood College**
*   Machine Learning for Bioinformatics (BIFX 546), Spring 2026
*   Instructor: Dr. Sarangan (Ravi) Ravichandran
****
**Dataset**

* Name: longevity-db/mouse-glioblastoma-snRNAseq

* Source: https://huggingface.co/datasets/longevity-db/mouse-glioblastoma-snRNAseq | CELLxGENE | GEO: GSE186252
****
**Project Goals**

Can we predict whether a cell belongs to the tumor core vs. the tumor microenvironment using transcriptomic summary features and QC-derived metrics?

While much research focuses on the tumor's genetic mutations, the tumor microenvironment (TME) remains under-explored. This dataset is unique because it provides single-nucleus RNA sequencing (snRNA-seq) across many cell types and from the tumor core itself and the surrounding microenvironment. This allows for a granular look at how the TME modulates cancer progression and immune evasion. The TME is a risk factor for multiple cancer types, in this dataset, the cancer type of the samples used was glioblastoma (GBM).

# Project Overview
##  1. Setup
##  2. Data Loading
##  3. Quality Control
##  4. Summary Statistics
##  5. Dimensionality Reduction
##  6. Clustering
##  7. Modeling
##  8. Interpretation
##  9. Next Steps
## 10. Acknowledgements

## 1. Setup

In [ ]:
pip install datasets

In [ ]:
from google.colab import userdata
HF_TOKEN = userdata.get('HF_TOKEN')

## 2. Data Loading, 10k Samples

In [ ]:
from datasets import load_dataset
import pandas as pd

dataset_name = "longevity-db/mouse-glioblastoma-snRNAseq"

streamed_dataset = load_dataset(dataset_name, streaming=True)

sample_size = 10000
sampled_data = []

for i, example in enumerate(streamed_dataset['train']):
    if i >= sample_size:
        break
    sampled_data.append(example)

df = pd.DataFrame(sampled_data)

print(df.shape)
df.head()

## 3. Quality Control
### Quality check
* Make sure to check for low counts and reads to eliminate bad samples

In [ ]:
# Standard single-cell QC filters
df = df[
    (df["nFeature_RNA"] > 200) &
    (df["nCount_RNA"] > 500)
].copy()

print("After QC:", df.shape)

### Classification
* redefine target variable to distinguish between tumor and non-tumor

In [ ]:
# Binary classification: tumor vs microenvironment
df["target"] = df["sub_celltype"].apply(
    lambda x: "tumor" if "tumor" in str(x).lower() else "non_tumor"
)

df["target"].value_counts()

### Exploratory Data Analysis (EDA)

distrubution plots

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(10,4))

sns.histplot(df["nCount_RNA"], bins=50, ax=axes[0])
axes[0].set_title("nCount_RNA")

sns.histplot(df["nFeature_RNA"], bins=50, ax=axes[1])
axes[1].set_title("nFeature_RNA")

plt.tight_layout()
plt.show()

Cell Type Proportions

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Plot for Tumor category
df[df['target'] == 'tumor']['sub_celltype'].value_counts().plot(kind='bar', ax=axes[0])
axes[0].set_title('Distribution of Tumor Cells')
axes[0].set_ylabel('Count')

# Plot for all other cell types
df[df['target'] == 'non_tumor']['sub_celltype'].value_counts().plot(kind='bar', ax=axes[1])
axes[1].set_title('Distribution of Non-Tumor Cell Types')
axes[1].set_ylabel('Count')

plt.tight_layout()
plt.show()

Target Balance

In [ ]:
sns.countplot(x="target", data=df)
plt.title("Tumor vs Microenvironment")
plt.show()

### Normalization

In [ ]:
import numpy as np

# Normalize counts per cell
df["nCount_RNA_norm"] = df["nCount_RNA"] / df["nCount_RNA"].sum() * 1e4
df["nCount_RNA_log"] = np.log1p(df["nCount_RNA_norm"])

df["nFeature_RNA_log"] = np.log1p(df["nFeature_RNA"])

## 4. Summary Statistics


In [ ]:
# Summary Stats table for the 10k dataset
df.describe().drop('count')

## 5. Dimensionality Reduction
### PCA + UMAP

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
import umap

features = [
    "nCount_RNA_log",
    "nFeature_RNA_log"
]

X = df[features]

# Scale
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# PCA
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

# UMAP
umap_model = umap.UMAP(random_state=42)
X_umap = umap_model.fit_transform(X_scaled)

In [ ]:
plt.figure(figsize=(8,6))

sns.scatterplot(
    x=X_pca[:,0],
    y=X_pca[:,1],
    hue=df["target"],
    palette={"tumor":"red","non_tumor":"blue"},
    s=10
)

plt.title("PCA: Tumor vs Microenvironment")
plt.xlabel("Principal Component 1")
plt.ylabel("Principal Component 2")
plt.show()

UMAP

By Cell Type

In [ ]:
plt.figure(figsize=(8,10))

sns.scatterplot(
    x=X_umap[:,0],
    y=X_umap[:,1],
    hue=df["sub_celltype"],
    s=10,
    linewidth=0
)

plt.title("UMAP by Cell Type")
plt.legend(loc='upper center', bbox_to_anchor=(0.5, -0.15), fancybox=True, shadow=True, ncol=3)
plt.tight_layout()
plt.show()

By Tumor vs Microenvironment

In [ ]:
plt.figure(figsize=(8,6))

sns.scatterplot(
    x=X_umap[:,0],
    y=X_umap[:,1],
    hue=df["target"],
    palette={"tumor":"red","non_tumor":"blue"},
    s=10
)

plt.title("UMAP: Tumor vs Microenvironment")
plt.show()

## 6. Clustering
### K means clustering from log-transformed counts and reads

In [ ]:
from sklearn.cluster import KMeans
import matplotlib.pyplot as plt
import seaborn as sns

# K means clustering
kmeans = KMeans(n_clusters=5, random_state=42, n_init='auto')
df["cluster"] = kmeans.fit_predict(X_scaled)

# Violin plot of cluster sizes
plt.figure(figsize=(8, 6))
sns.violinplot(x="cluster", y="nCount_RNA_log", data=df, palette='viridis', hue="cluster", legend=False)
plt.title("Distribution of nCount_RNA_log per Cluster")
plt.xlabel("Cluster")
plt.ylabel("Log-normalized RNA Count")
plt.show()

# Visualization of clusters with UMAP
plt.figure(figsize=(10, 8))
sns.scatterplot(
    x=X_umap[:, 0],
    y=X_umap[:, 1],
    hue=df["cluster"].astype(str), # Convert to string for discrete colors
    palette='viridis',
    s=10,
    alpha=0.7
)

# plot options
plt.title("UMAP Plot with KMeans Clusters")
plt.xlabel("UMAP 1")
plt.ylabel("UMAP 2")
plt.legend(title="Cluster", bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()

Cluster Interpretation

In [ ]:
pd.crosstab(df["cluster"], df["target"])

## 7. Modeling
### Training and Testing

In [ ]:
from sklearn.model_selection import train_test_split

y = df["target"]

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

### Random Forest Model & Decision trees

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score
import matplotlib.pyplot as plt
from sklearn.tree import plot_tree

rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)

y_pred = rf.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

# Visualize one of the decision trees from the Random Forest
plt.figure(figsize=(20, 10))
plot_tree(rf.estimators_[0],
          feature_names=features,
          class_names=rf.classes_, # Use classes from the model
          filled=True,
          rounded=True,
          fontsize=8)
plt.title("Decision Tree from Random Forest")
plt.show()

### Feature Importance

In [ ]:
import pandas as pd

importance_df = pd.DataFrame({
    "feature": features,
    "importance": rf.feature_importances_
}).sort_values(by="importance", ascending=False)

sns.barplot(data=importance_df, x="importance", y="feature")
plt.title("Feature Importance")
plt.show()